In [7]:
# =============================================================================
# NOTEBOOK: load_fact.ipynb
#
# DESCRIPCIÓN:
#   Notebook genérico para cargar una tabla de hechos en la capa Gold.
#   Aplica cargas incrementales, transforma datos y realiza lookups de claves.
#
# PARÁMETROS:
#   - task_id (integer): El ID de la tarea a ejecutar, definido en la tabla de control.
# =============================================================================

# --- 1. Importaciones y Configuración ---

from pyspark.sql.functions import row_number, col, lit, coalesce, expr, max as spark_max
from datetime import datetime
from pyspark.sql.window import Window
from notebookutils import mssparkutils
from functools import reduce  # Necesario para los JOINs con claves compuestas
import json
import time
import random

# Configuración para LEER fechas antiguas de fuentes Parquet/Delta
spark.conf.set("spark.sql.parquet.datetimeRebaseModeInRead", "LEGACY")

# Configuración para ESCRIBIR fechas antiguas a destinos Parquet/Delta (RECOMENDADA)
spark.conf.set("spark.sql.parquet.datetimeRebaseModeInWrite", "CORRECTED")

# --- Parámetros ---
task_id_param = 25
# --- Configuración Global ---
control_table_full_name = "lh_silver_shortcuts.dbo.silver_to_gold_control" 
current_utc_timestamp = datetime.utcnow()


StatementMeta(, dddd842f-d085-41cc-bd48-ac473569ffb7, 9, Finished, Available, Finished)

In [8]:
print(f"--- Iniciando Carga de Tabla de Hechos para Tarea ID: {task_id_param} ---")

# --- Control de concurrencia ---
MAX_RETRIES = 10
BASE_DELAY_SEC = 3
MAX_DELAY_SEC = 40

def is_concurrent_error(ex: Exception) -> bool:
    """Detecta si el error es por concurrencia en Delta Lake."""
    msg = str(ex).lower()
    return "concurrentappendexception" in msg or ("concurrent" in msg and "delta" in msg)

def run_delta_operation_with_retry(fn, operation_name: str = "operación Delta") -> None:
    """Ejecuta una operación Delta (MERGE, write, etc.) con reintentos ante ConcurrentAppendException."""
    last_error = None
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            fn()
            if attempt > 1:
                print(f"   {operation_name} completada (intento {attempt}).")
            return
        except Exception as e:
            last_error = e
            if attempt < MAX_RETRIES and is_concurrent_error(e):
                delay = min(BASE_DELAY_SEC * (2 ** (attempt - 1)) + random.uniform(0, 1), MAX_DELAY_SEC)
                print(f"   Concurrencia detectada en {operation_name} (intento {attempt}/{MAX_RETRIES}). Reintento en {delay:.1f}s...")
                time.sleep(delay)
            else:
                raise
    if last_error is not None:
        raise last_error

# --- 2. Función de Logging ---
def update_task_status(status, message, new_watermark=None):
    """
    Actualiza la tabla de control (con reintentos ante concurrencia).
    NOTA: spark.sql("UPDATE...") solo funciona si la tabla de control es Delta.
    """
    safe_message = message.replace("'", "''")
    watermark_update_sql = ""
    if new_watermark is not None:
        watermark_str = new_watermark.strftime('%Y-%m-%d %H:%M:%S.%f') if isinstance(new_watermark, datetime) else str(new_watermark)
        watermark_update_sql = f", last_watermark_value = '{watermark_str}'"

    update_query = f"""
        UPDATE {control_table_full_name}
        SET
            last_run_status = '{status}',
            last_message = '{safe_message}',
            last_run_at = CAST('{current_utc_timestamp}' AS TIMESTAMP),
            updated_at = CAST('{current_utc_timestamp}' AS TIMESTAMP)
            {watermark_update_sql}
        WHERE
            task_id = {task_id_param}
    """

    last_error = None
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            spark.sql(update_query)
            print(f"Log actualizado: Estado='{status}', Mensaje='{message}'" + (f" (intento {attempt})" if attempt > 1 else ""))
            return
        except Exception as e:
            last_error = e
            if attempt < MAX_RETRIES and is_concurrent_error(e):
                delay = min(BASE_DELAY_SEC * (2 ** (attempt - 1)) + random.uniform(0, 1), MAX_DELAY_SEC)
                print(f"   Concurrencia detectada al actualizar control (intento {attempt}/{MAX_RETRIES}). Reintento en {delay:.1f}s...")
                time.sleep(delay)
            else:
                print(f"FATAL: No se pudo actualizar la tabla de control. Razón: {e}")
                raise
    if last_error is not None:
        raise last_error

# --- 3. Bloque Principal de Ejecución ---
source_df = None
try:
    # 3.1. LEER METADATOS DE LA TAREA
    print(f"Paso 1: Leyendo metadatos de la tarea desde '{control_table_full_name}'...")
    config_df = spark.sql(f"SELECT * FROM {control_table_full_name} WHERE task_id = {task_id_param} AND is_enabled = true")
    if config_df.rdd.isEmpty():
        raise ValueError(f"Tarea ID '{task_id_param}' no encontrada o está deshabilitada.")
    config = config_df.first()
    source_object_full_name = f'{config["source_lakehouse"]}.{config["source_schema"]}.{config["source_table"]}'
    target_lakehouse = config["target_lakehouse"]
    target_schema = config["target_schema"]
    target_table = config["target_table"]
    target_table_full_name = f"{target_lakehouse}.{target_schema}.{target_table}"
    business_keys_list = [key.strip() for key in config["business_keys"].split(',')]
    watermark_column = config["watermark_column"]
    last_watermark_value = config["last_watermark_value"]
    column_mapping_json = config["column_mapping_json"]
    print("Metadatos leídos correctamente.")

    # 3.2. LEER DATOS DE ORIGEN (INCREMENTAL)
    print(f"Paso 2: Leyendo datos de origen de '{source_object_full_name}'...")
    source_query = f"SELECT * FROM {source_object_full_name}"
    if watermark_column and last_watermark_value:
        print(f"Aplicando filtro incremental: {watermark_column} > '{last_watermark_value}'")
        source_query += f" WHERE {watermark_column} > '{last_watermark_value}'"
    source_df = spark.sql(source_query)
    if source_df.rdd.isEmpty():
        print("No hay registros nuevos para procesar.")
        update_task_status('Success', 'No new records to process.')
        mssparkutils.notebook.exit("No new records.")
    source_df.cache()
    new_watermark = source_df.agg({watermark_column: "max"}).collect()[0][0]
    print(f"Carga incremental desde '{source_object_full_name}'. Registros leídos: {source_df.count()}. Nuevo watermark: {new_watermark}")

    # 3.3. TRANSFORMAR DATOS: LOOKUPS Y SELECCIÓN
    print("Paso 3: Iniciando transformación de datos y lookups a dimensiones...")
    mappings = json.loads(column_mapping_json)
    transformed_df = source_df
    final_select_cols_mapping = []
    
    lookups = [m for m in mappings if m['Type'] == 'LOOKUP']
    for i, lookup in enumerate(lookups):
        dim_alias = f"d{i}"
        source_parts = lookup['Source'].split(';')
        dim_table_str = source_parts[0]
        
        if len(source_parts) == 2:
            fact_keys = [key.strip() for key in source_parts[1].split(',')]
            dim_keys = fact_keys
        elif len(source_parts) == 3:
            fact_keys = [key.strip() for key in source_parts[1].split(',')]
            dim_keys = [key.strip() for key in source_parts[2].split(',')]
        else:
            raise ValueError(f"Formato de 'Source' incorrecto para el lookup: {lookup['Source']}")

        #added by oce
        target_parts = lookup['Target'].split(';')
        if len(target_parts) == 1:
            surrogate_key = target_parts[0]
            new_surrogate_key_name = surrogate_key
        elif len(target_parts) == 2:
            surrogate_key = target_parts[0]
            new_surrogate_key_name = target_parts[1]
        else:
            raise ValueError(f"Formato de 'Target' incorrecto para el lookup: {lookup['Target']}")
        #added by oce

        print(f"surrogate_key: '{surrogate_key}' new_surrogate_key_name: '{new_surrogate_key_name}' ")
        print(f"fact_keys: '{fact_keys[0]}', '{fact_keys[1]}'")
        print(f"dim_keys: '{dim_keys[0]}','{dim_keys[1]}")

        #dim_table_full_name = f"{target_lakehouse}.{target_schema}.{dim_table_str}"
        dim_table_full_name = f"{target_lakehouse}.{dim_table_str}"
        dim_df = spark.table(dim_table_full_name).cache()
        #surrogate_key = lookup['Target']

        if len(fact_keys) != len(dim_keys):
            raise ValueError(f"Error en el lookup para {dim_table_str}: El número de claves no coincide.")
            
        join_conditions_list = [transformed_df[fact_keys[j]] == col(f"{dim_alias}.{dim_keys[j]}") for j in range(len(fact_keys))]
        join_condition = reduce(lambda a, b: a & b, join_conditions_list)

        #print(f"join_condition: '{join_condition}'")
        
        print(f"Realizando lookup a '{dim_table_str}' con alias '{dim_alias}'...")
        
        # ========================================================================
        # === SOLUCIÓN DEFINITIVA: REFERENCIA EXPLÍCITA AL DATAFRAME ORIGINAL ===
        # ========================================================================
        # 1. Guardar una referencia al DataFrame ANTES del join. Sus columnas no son ambiguas.
        df_before_join = transformed_df
        
        # 2. Realizar el join, que crea un DataFrame ancho y temporal con columnas ambiguas.
        joined_df = df_before_join.join(
            dim_df.alias(dim_alias),
            join_condition,
            "left_outer"
        )

        # 3. Construir la lista de selección haciendo referencia explícita
        #    a las columnas del DataFrame de ANTES del join (df_before_join[c]).

        #new_surrogate_key_name = lookup['Target']

        select_expressions = [df_before_join[c] for c in df_before_join.columns] + [
            coalesce(col(f"{dim_alias}.{surrogate_key}").alias(new_surrogate_key_name), lit(-1)).alias(new_surrogate_key_name)
        ]
        #print(select_expressions)
        # 4. Recrear el DataFrame a partir de la selección explícita y no ambigua.
        #    Esto descarta todas las columnas de la dimensión excepto la surrogate key.
        transformed_df = joined_df.select(*select_expressions)
        print(f"Join con '{dim_alias}' completado y ambigüedad resuelta.")
        # ========================================================================

    # Construir el mapeo de columnas final para reemplazar las claves de negocio por las subrogadas
    for mapping in mappings:
        if mapping['Type'] == 'LOOKUP':
            #final_select_cols_mapping.append(col(mapping['Target']))

            #added by oce
            target_parts = mapping['Target'].split(';')
            if len(target_parts) == 1:
                surrogate_key = target_parts[0]
                new_surrogate_key_name = surrogate_key
            elif len(target_parts) == 2:
                surrogate_key = target_parts[0]
                new_surrogate_key_name = target_parts[1]
            else:
                raise ValueError(f"Formato de 'Target' incorrecto para el lookup: {mapping['Target']}")
            #added by oce

            final_select_cols_mapping.append(col(new_surrogate_key_name))
        else:
            final_select_cols_mapping.append(col(mapping['Source']).alias(mapping['Target']))

    final_df = transformed_df.select(*final_select_cols_mapping)
    print("Transformación de datos completada.")

    #print(final_select_cols_mapping)
    
    # 3.4. CREAR TABLA DE HECHOS SI NO EXISTE
    if not spark.catalog.tableExists(target_table_full_name):
        print(f"Paso 4: La tabla destino '{target_table_full_name}' no existe. Creándola...")
        final_df.limit(0).write.format("delta").saveAsTable(target_table_full_name)
        print("Tabla de hechos creada exitosamente.")
    else:
        print(f"Paso 4: La tabla destino '{target_table_full_name}' ya existe.")

    # ========================================================================
    # === INICIO: Bloque de Deduplicación Antes del MERGE ===
    # ========================================================================
    print("Aplicando paso de deduplicación final para garantizar unicidad en la clave de MERGE...")

    # Este bloque usa 'business_keys_list' y 'watermark_column' leídos de los metadatos.
    # 'business_keys_list' debe contener las claves subrogadas (ej. 'account_key', 'company_key').
    # 'watermark_column' (ej. 'last_updated_at') decide qué registro se conserva en caso de duplicados.

    print(f"bussines_keys_list: {business_keys_list}")
    
    window_spec = Window.partitionBy(*business_keys_list).orderBy(col(watermark_column).desc())
    final_df_with_rownum = final_df.withColumn("row_num", row_number().over(window_spec))
    deduplicated_final_df = final_df_with_rownum.filter(col("row_num") == 1).drop("row_num")
    
    # Opcional: Imprimir si se encontraron y eliminaron duplicados
    original_count = final_df.count()
    deduplicated_count = deduplicated_final_df.count()

    if original_count > deduplicated_count:
        print(f"ADVERTENCIA: Se detectaron y eliminaron {original_count - deduplicated_count} duplicados antes del MERGE.")

    # Re-asignar el DataFrame limpio a la variable final_df para que el resto del script lo utilice
    final_df = deduplicated_final_df

    # ========================================================================
    # === FIN: Bloque de Deduplicación ===
    # ========================================================================
    
    # 3.5. CARGAR DATOS EN LA TABLA DE HECHOS (MERGE)
    print(f"Paso 5: Iniciando operación MERGE en '{target_table_full_name}'...")
    final_df.createOrReplaceTempView("source_view_for_merge")
    merge_on_condition = " AND ".join([f"Target.{key} = Source.{key}" for key in business_keys_list])
    update_set = ", ".join([f"Target.{c} = Source.{c}" for c in final_df.columns if c not in business_keys_list])
    insert_cols = ", ".join(final_df.columns)
    insert_values = ", ".join([f"Source.{c}" for c in final_df.columns])
    merge_sql = f"""
        MERGE INTO {target_table_full_name} Target
        USING source_view_for_merge Source ON {merge_on_condition}
        WHEN MATCHED THEN UPDATE SET {update_set}
        WHEN NOT MATCHED THEN INSERT ({insert_cols}) VALUES ({insert_values})
    """

    def _do_merge():
        spark.sql(merge_sql)
    run_delta_operation_with_retry(_do_merge, f"MERGE en {target_table_full_name}")
    print(f"MERGE en '{target_table_full_name}' completado.")

    # 3.6. REGISTRAR ÉXITO Y NUEVO WATERMARK
    print("Paso 6: Registrando éxito en la tabla de control...")
    update_task_status('Success', f'Task completed successfully. Processed {final_df.count()} records.', new_watermark)
    
except Exception as e:
    error_message = str(e).replace('\n', ' ').replace('\r', '')
    print(f"FATAL ERROR: Ocurrió una excepción durante la ejecución. {error_message}")
    update_task_status('Failed', error_message)
    raise e
finally:
    if source_df:
        source_df.unpersist()
    print("--- Proceso Finalizado ---")

StatementMeta(, dddd842f-d085-41cc-bd48-ac473569ffb7, 10, Finished, Available, Finished)

--- Iniciando Carga de Tabla de Hechos para Tarea ID: 25 ---
Paso 1: Leyendo metadatos de la tarea desde 'lh_silver_shortcuts.dbo.silver_to_gold_control'...
Metadatos leídos correctamente.
Paso 2: Leyendo datos de origen de 'lh_silver_shortcuts.sd.sales'...
Aplicando filtro incremental: last_updated_at > '1900-01-01'
Carga incremental desde 'lh_silver_shortcuts.sd.sales'. Registros leídos: 5365500. Nuevo watermark: 2025-12-08 12:10:07.107679
Paso 3: Iniciando transformación de datos y lookups a dimensiones...
surrogate_key: 'vendor_key' new_surrogate_key_name: 'vendor_customer_key' 
fact_keys: 'vendor_customer_id', 'company_code'
dim_keys: 'address_id','company_code
Realizando lookup a 'dimentions.dim_vendors' con alias 'd0'...
Join con 'd0' completado y ambigüedad resuelta.
surrogate_key: 'code_key' new_surrogate_key_name: 'sales_office_key' 
fact_keys: 'sales_office_id', 'company_code'
dim_keys: 'code_value','company_code
Realizando lookup a 'dimentions.dim_sales_office' con alias 'd